# 📍 Notebook 04 — Georeferencing

**Project:** Texas Injection Wells — Delaware Basin Analysis  
**Author:** Juan David Antolinez  
**Purpose:** Attach geographic coordinates (latitude/longitude) to each well in the clean injection dataset by merging with the well inventory generated in Notebook 01.

---

### Why This Step?
The H10 scraper (Notebook 02) collects injection data but **does not include coordinates** — the RRC H10 portal does not expose them. Coordinates were collected separately via the Texas RRC open data API (Socrata) in Notebook 01.

This notebook joins both datasets by `UicNumber` using a LEFT JOIN to preserve all injection records.

### Join Logic
| Table | Key Column | Records |
|---|---|---|
| `resultados_h10_clean.xlsx` | `UicNumber` | 175,838 rows |
| `wells_delaware_coordinates.csv` | `uic_number` | 2,875 wells |

### Output
- `data/processed/delaware_injection_final.xlsx` — fully georeferenced dataset, ready for Power BI

## 1. Library Imports

In [1]:
import pandas as pd
from pathlib import Path

## 2. Project Paths

In [2]:
BASE_DIR          = Path.cwd().parent
DATA_CLEAN        = BASE_DIR / "data" / "processed" / "resultados_h10_clean.xlsx"
WELLS_COORDINATES = BASE_DIR / "data" / "raw"       / "wells_delaware_coordinates.csv"
DATA_FINAL        = BASE_DIR / "data" / "processed" / "delaware_injection_final.xlsx"

## 3. Load Datasets

In [3]:
data_clean_df   = pd.read_excel(DATA_CLEAN)
coordinates_df  = pd.read_csv(WELLS_COORDINATES)

print(f"Injection data:  {data_clean_df.shape[0]:,} rows × {data_clean_df.shape[1]} columns")
print(f"Coordinates:     {coordinates_df.shape[0]:,} rows × {coordinates_df.shape[1]} columns")
print()
print(f"Coordinates columns: {coordinates_df.columns.tolist()}")

## 4. Merge Coordinates

Join the coordinates table to the injection dataset on `UicNumber`.  
Using a **LEFT JOIN** to preserve all 175,838 injection records — even if a well has no matching coordinates.

In [4]:
# Merge on UIC number
data_clean_df = data_clean_df.merge(
    coordinates_df[['uic_number', 'latitude_nad83', 'longitude_nad83']],
    left_on  = 'UicNumber',
    right_on = 'uic_number',
    how      = 'left'
)

# Populate coordinate columns
data_clean_df['GisLatNad83']  = data_clean_df['latitude_nad83']
data_clean_df['GisLongNad83'] = data_clean_df['longitude_nad83']

# Drop temporary merge columns
data_clean_df = data_clean_df.drop(columns=['latitude_nad83', 'longitude_nad83', 'uic_number'])

print(f"Rows after merge:          {len(data_clean_df):,}")
print(f"GisLatNad83 null values:   {data_clean_df['GisLatNad83'].isnull().sum():,}")
print(f"GisLongNad83 null values:  {data_clean_df['GisLongNad83'].isnull().sum():,}")

## 5. Save Final Dataset

This is the fully processed, georeferenced dataset — ready for Power BI and EDA.

In [5]:
data_clean_df.to_excel(DATA_FINAL, index=False)

print(f"✅  Saved: {DATA_FINAL}")
print(f"    Rows:         {len(data_clean_df):,}")
print(f"    Columns:      {data_clean_df.shape[1]}")
print(f"    Unique wells: {data_clean_df['WellApi'].nunique():,}")